# Machine Learning: CV Recommender — Pembuatan Model

Notebook ini menunjukkan proses dari data mentah lowongan kerja sampai model pencarian lowongan siap pakai.

**Alur:** Ambil data → Bersihkan teks → Ubah jadi vektor → Bangun index pencarian → Evaluasi → Simpan model

## Langkah 1: Import Library
Memuat semua pustaka yang dibutuhkan untuk proses pengolahan data dan pembuatan model.

In [1]:
# ==============================================================================
# LANGKAH 1: Import Library
# Tujuan: Memuat semua pustaka yang dibutuhkan
# ==============================================================================

# Mengimpor pandas untuk manipulasi data tabel (DataFrame)
import pandas as pd

# Mengimpor numpy untuk operasi array numerik (vektor embedding)
import numpy as np

# Mengimpor SentenceTransformer untuk mengubah teks menjadi vektor 384 dimensi
from sentence_transformers import SentenceTransformer

# Mengimpor faiss untuk pencarian kemiripan vektor secara cepat
import faiss

# Mengimpor re untuk pembersihan teks menggunakan Regular Expression
import re

# Mengimpor json untuk menyimpan metadata dalam format JSON
import json

# Mengimpor os untuk manipulasi path file
import os

# Mengimpor BigQuery client untuk mengambil data dari Google Cloud
from google.cloud import bigquery

print("Semua library berhasil dimuat!")

Semua library berhasil dimuat!


## Langkah 2: Ambil Data dari BigQuery
Mengambil seluruh data lowongan kerja dari tabel BigQuery. Data ini berisi ~108.000 baris lowongan dengan kolom seperti `job_title`, `job_description`, `skills_required`, `company_name`, dan `location`.

In [2]:
# ==============================================================================
# LANGKAH 2: Ambil Data dari BigQuery
# Tujuan: Mengambil dataset lowongan kerja dari database cloud GCP
# ==============================================================================

def ambil_data_lowongan(project_id, dataset_id, table_id):
    """
    Mengambil seluruh data lowongan dari tabel BigQuery.

    Parameter:
        project_id (str) -- ID project Google Cloud
        dataset_id (str) -- ID dataset di BigQuery
        table_id (str) -- ID tabel yang berisi data lowongan

    Return:
        DataFrame -- seluruh baris data lowongan kerja
    """
    client = bigquery.Client(project=project_id)
    # Query sederhana: ambil semua kolom dan baris dari tabel lowongan
    query = f"SELECT * FROM `{project_id}.{dataset_id}.{table_id}`"
    df = client.query(query).to_dataframe()
    return df

# Ambil data dari BigQuery
df = ambil_data_lowongan(
    project_id="deductive-reach-443812-q3",
    dataset_id="pnm_jobs",
    table_id="unified_jobs"
)

print(f"Total data lowongan: {len(df)} baris")
print(f"Kolom yang tersedia: {list(df.columns)}")
df.head(3)

C:\Users\ACER\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Total data lowongan: 108963 baris
Kolom yang tersedia: ['unique_id', 'data_source', 'job_title', 'company_name', 'job_description', 'location', 'min_salary', 'max_salary', 'contract_time', 'contract_type', 'experience_level', 'skills_required']


,unique_id,data_source,job_title,company_name,job_description,location,min_salary,max_salary,contract_time,contract_type,experience_level,skills_required
0,KAG_3884431614,Kaggle_LinkedIn,Accounts Payable Clerk - Temp to Hire,None,Accounts Payable Staff Job Description\nAP sta...,"Irvine, CA",25.0,27.0,TEMPORARY,Not Specified,Associate,None
1,KAG_3884432782,Kaggle_LinkedIn,Staff Accountant,None,Now Hiring - Staff Accountant Full-Time – Sala...,"Islandia, NY",65000.0,75000.0,FULL_TIME,Not Specified,Associate,None
2,KAG_3884808345,Kaggle_LinkedIn,Associate Attorney - Estate Planning,None,"About the Company: Hall Law Firm, P.C. is a bo...","San Francisco, CA",NaN,NaN,FULL_TIME,Not Specified,Associate,None


## Langkah 3: Bersihkan Teks
Teks deskripsi pekerjaan dari internet sering mengandung kode HTML, URL, dan email yang harus dihapus. Namun, simbol penting untuk nama teknologi (`c++`, `c#`, `.net`) harus **dipertahankan** agar model bisa mengenalinya.

In [3]:
# ==============================================================================
# LANGKAH 3: Bersihkan Teks
# Tujuan: Membersihkan teks dari gangguan (HTML, URL) tapi mempertahankan
#         simbol penting untuk nama teknologi
# ==============================================================================

def bersihkan_teks(teks):
    """
    Membersihkan teks deskripsi pekerjaan dari noise.

    Parameter:
        teks (str) -- teks mentah dari deskripsi lowongan

    Return:
        str -- teks yang sudah bersih dan siap diproses model
    """
    if pd.isna(teks):
        return ""
    teks = str(teks).lower()

    # Hapus tag HTML seperti <div>, <br>, <p>, dll.
    teks = re.sub(r"<[^>]+>", " ", teks)

    # Hapus URL (http://... atau www....)
    teks = re.sub(r"https?://\S+|www\.\S+", " ", teks)

    # Hapus alamat email
    teks = re.sub(r"\S+@\S+\.\S+", " ", teks)

    # Hapus karakter khusus KECUALI simbol penting: +, #, ., -, /
    # Ini mempertahankan nama teknologi seperti c++, c#, .net, ci/cd
    teks = re.sub(r"[^a-z0-9\s\+\#\.\-\/]", " ", teks)

    # Hapus spasi berlebih
    teks = re.sub(r"\s+", " ", teks).strip()
    return teks

# Contoh penggunaan
contoh = "<p>Looking for a <b>C++ Developer</b> with .NET experience. Email: hr@company.com</p>"
print(f"Sebelum: {contoh}")
print(f"Sesudah: {bersihkan_teks(contoh)}")

Sebelum: <p>Looking for a <b>C++ Developer</b> with .NET experience. Email: hr@company.com</p>
Sesudah: looking for a c++ developer with .net experience. email


## Langkah 4: Gabungkan Kolom dan Buang Data Kosong
Menggabungkan judul, deskripsi, dan skill menjadi satu kolom `combined_text`. Deskripsi dipotong 500 karakter karena model Sentence-BERT memiliki batas panjang input (token limit) — 500 karakter awal sudah cukup mewakili informasi utama pekerjaan.

In [4]:
# ==============================================================================
# LANGKAH 4: Gabungkan Kolom dan Buang Data Kosong
# Tujuan: Membuat satu kolom teks gabungan yang merepresentasikan setiap lowongan
# ==============================================================================

# Buang baris yang tidak punya judul atau deskripsi (data tidak lengkap)
df = df.dropna(subset=["job_title", "job_description"])
print(f"Jumlah baris setelah buang data kosong: {len(df)}")

# Bersihkan teks judul dan deskripsi
df["job_title_clean"] = df["job_title"].apply(bersihkan_teks)
df["job_description_clean"] = df["job_description"].apply(bersihkan_teks)

# Isi kolom skills_required yang kosong dengan string kosong
df["skills_required"] = df["skills_required"].fillna("")

# Gabungkan: judul + deskripsi (maks 500 karakter) + skill
# Pembatasan 500 karakter menghemat memori dan sesuai batas token model BERT
df["combined_text"] = (
    df["job_title_clean"] + " " +
    df["job_description_clean"].str[:500] + " " +
    df["skills_required"].astype(str)
)

# Buang baris yang combined_text-nya terlalu pendek (kurang informatif)
df = df[df["combined_text"].str.len() > 30].reset_index(drop=True)
print(f"Jumlah baris final: {len(df)}")

Jumlah baris setelah buang data kosong: 108963


Jumlah baris final: 108940


## Langkah 5: Ubah Teks Jadi Vektor (Embedding)
Model Sentence-BERT mengubah setiap teks lowongan menjadi **vektor 384 dimensi** — yaitu deret 384 angka desimal yang merepresentasikan makna teks. Vektor dinormalisasi agar panjangnya = 1, sehingga perkalian titik (dot product) antar vektor sama dengan **cosine similarity** (kemiripan arah).

In [5]:
# ==============================================================================
# LANGKAH 5: Ubah Teks Jadi Vektor (Embedding)
# Tujuan: Mengubah setiap combined_text menjadi vektor numerik 384 dimensi
# ==============================================================================

# Muat model Sentence-BERT ringan (~120MB)
model = SentenceTransformer("all-MiniLM-L6-v2")

# Encode semua teks gabungan menjadi vektor
# normalize_embeddings=True: vektor dinormalisasi sehingga dot product = cosine similarity
# show_progress_bar=True: menampilkan progress bar saat encoding
embeddings = model.encode(
    df["combined_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=64
)

# Konversi ke float32 karena FAISS membutuhkan tipe data ini
embeddings = np.array(embeddings).astype(np.float32)

print(f"Bentuk matriks embedding: {embeddings.shape}")
print(f"Artinya: {embeddings.shape[0]} lowongan, masing-masing punya {embeddings.shape[1]} dimensi")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1703 [00:00<?, ?it/s]

Bentuk matriks embedding: (108940, 384)
Artinya: 108940 lowongan, masing-masing punya 384 dimensi


## Langkah 6: Bangun Index FAISS
FAISS `IndexFlatIP` (Inner Product) menyimpan semua vektor dan memungkinkan pencarian tetangga terdekat secara instan. Karena vektor sudah dinormalisasi, **Inner Product = Cosine Similarity** — sehingga pencarian berdasarkan kemiripan arah vektor.

In [6]:
# ==============================================================================
# LANGKAH 6: Bangun Index FAISS
# Tujuan: Membuat struktur data pencarian cepat untuk semua vektor lowongan
# ==============================================================================

# Dimensi vektor = 384 (sesuai output model all-MiniLM-L6-v2)
dimensi = embeddings.shape[1]

# FlatIP = pencarian brute-force menggunakan Inner Product (cosine similarity)
index = faiss.IndexFlatIP(dimensi)

# Masukkan semua vektor lowongan ke dalam index
index.add(embeddings)

print(f"Index FAISS berhasil dibangun!")
print(f"Total vektor di index: {index.ntotal}")
print(f"Dimensi per vektor: {dimensi}")

Index FAISS berhasil dibangun!
Total vektor di index: 108940
Dimensi per vektor: 384


## Langkah 7: Evaluasi Sederhana (Precision@5)
Kita menguji apakah model bisa menemukan lowongan yang relevan. Caranya: ambil 50 lowongan acak sebagai query, cari 5 lowongan terdekat, lalu cek berapa persen yang judulnya serupa (relevan). **Precision@5** = proporsi lowongan relevan dari 5 hasil teratas.

In [7]:
# ==============================================================================
# LANGKAH 7: Evaluasi Sederhana (Precision@5)
# Tujuan: Mengukur seberapa akurat model dalam menemukan lowongan relevan
# ==============================================================================

def hitung_precision_at_k(df, index, model, jumlah_sample=50, k=5):
    """
    Menghitung rata-rata Precision@K menggunakan sample acak.

    Parameter:
        df (DataFrame) -- data lowongan dengan kolom combined_text dan job_title
        index (faiss.Index) -- index FAISS yang sudah terisi vektor
        model (SentenceTransformer) -- model untuk mengubah teks jadi vektor
        jumlah_sample (int) -- berapa lowongan yang diuji sebagai query
        k (int) -- jumlah hasil teratas yang diperiksa

    Return:
        float -- rata-rata Precision@K (0.0 - 1.0)
    """
    # Ambil sample acak dari dataset
    sample = df.sample(n=min(jumlah_sample, len(df)), random_state=42)
    skor_list = []

    for idx, row in sample.iterrows():
        # Ubah teks query menjadi vektor
        query_vec = model.encode(
            [row["combined_text"]], normalize_embeddings=True
        ).astype(np.float32)

        # Cari k+1 terdekat (karena hasil pertama biasanya dirinya sendiri)
        _, indices = index.search(query_vec, k + 1)

        # Ambil judul query sebagai acuan relevansi
        judul_query = str(row["job_title"]).lower().split()[0]  # kata pertama judul

        # Hitung berapa hasil yang judulnya mengandung kata kunci yang sama
        relevan = 0
        diperiksa = 0
        for res_idx in indices[0]:
            # Lewati dirinya sendiri
            if res_idx == idx or res_idx >= len(df):
                continue
            judul_hasil = str(df.iloc[res_idx]["job_title"]).lower()
            # Hasil dianggap relevan jika kata kunci judul query muncul di judul hasil
            if judul_query in judul_hasil:
                relevan += 1
            diperiksa += 1
            if diperiksa >= k:
                break

        # Precision = jumlah relevan / jumlah yang diperiksa
        precision = relevan / max(diperiksa, 1)
        skor_list.append(precision)

    return np.mean(skor_list)

# Jalankan evaluasi
precision = hitung_precision_at_k(df, index, model)
print(f"Precision@5 rata-rata: {precision:.2%}")
print(f"Artinya: dari 5 lowongan teratas yang direkomendasikan, ~{precision*5:.1f} di antaranya relevan.")

Precision@5 rata-rata: 52.80%
Artinya: dari 5 lowongan teratas yang direkomendasikan, ~2.6 di antaranya relevan.


## Langkah 8: Simpan Artefak Model
Menyimpan index FAISS dan metadata ke folder `models/`. **Penting:** baris ke-N di `faiss_job_index.bin` harus selalu cocok dengan baris ke-N di `job_metadata.csv` — karena FAISS mengembalikan nomor indeks baris, dan kita perlu mengambil informasi lowongan dari CSV berdasarkan indeks tersebut.

In [8]:
# ==============================================================================
# LANGKAH 8: Simpan Artefak Model
# Tujuan: Menyimpan hasil pembuatan model agar bisa dipakai oleh streamlit_app.py
# ==============================================================================

# Buat folder models/ jika belum ada
os.makedirs("models", exist_ok=True)

# Simpan index FAISS ke file biner
faiss.write_index(index, "models/faiss_job_index.bin")
print(f"Index FAISS disimpan: models/faiss_job_index.bin ({index.ntotal} vektor)")

# Simpan metadata lowongan ke CSV
# Kolom yang disimpan: informasi yang dibutuhkan dashboard untuk menampilkan hasil
kolom_metadata = ["job_title", "company_name", "location", "job_description", "skills_required"]
kolom_tersedia = [k for k in kolom_metadata if k in df.columns]
df[kolom_tersedia].to_csv("models/job_metadata.csv", index=False)
print(f"Metadata lowongan disimpan: models/job_metadata.csv ({len(df)} baris)")

print("\nSemua artefak model berhasil disimpan!")
print("PENTING: Baris ke-N di index FAISS harus cocok dengan baris ke-N di metadata CSV.")

Index FAISS disimpan: models/faiss_job_index.bin (108940 vektor)


Metadata lowongan disimpan: models/job_metadata.csv (108940 baris)

Semua artefak model berhasil disimpan!
PENTING: Baris ke-N di index FAISS harus cocok dengan baris ke-N di metadata CSV.


## Langkah 9: Evaluasi Lanjutan (Akurasi Tanpa Retraining)
Kita dapat mengevaluasi akurasi model rekomendasi lowongan kerja secara langsung dengan memuat indeks FAISS dan data metadata yang sudah disimpan sebelumnya. Di sini kita memuat model, menguji pencarian kemiripan menggunakan sample acak dari metadata, dan menghitung metrik performa: **Precision@K** (seberapa relevan k lowongan teratas) dan **Mean Reciprocal Rank (MRR)** (posisi rekomendasi relevan pertama).

In [9]:
# ==============================================================================
# LANGKAH 9: Evaluasi Lanjutan (Akurasi Tanpa Retraining)
# Tujuan: Memuat model & index yang sudah tersimpan untuk dievaluasi akurasinya
# ==============================================================================

import os
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

# 1. Muat model dan metadata dari disk
print("Memuat model Sentence-BERT...")
model_eval = SentenceTransformer("all-MiniLM-L6-v2")

print("Memuat indeks FAISS dan metadata...")
index_eval = faiss.read_index("models/faiss_job_index.bin")
df_eval = pd.read_csv("models/job_metadata.csv")

# Buat kolom combined_text untuk simulasi query (judul + deskripsi dipotong + skill)
df_eval["skills_required"] = df_eval["skills_required"].fillna("")
df_eval["job_description_clean"] = df_eval["job_description"].fillna("").apply(lambda x: str(x).lower())
df_eval["combined_text"] = (
    df_eval["job_title"].fillna("").str.lower() + " " +
    df_eval["job_description_clean"].str[:500] + " " +
    df_eval["skills_required"].astype(str).str.lower()
)

def evaluasi_independen(df, index, model, jumlah_sample=100, k=5):
    """
    Menghitung Precision@K dan Mean Reciprocal Rank (MRR) secara langsung.
    """
    # Ambil sample acak
    np.random.seed(42)
    indices_sample = np.random.choice(len(df), size=min(jumlah_sample, len(df)), replace=False)
    
    precision_scores = []
    mrr_scores = []
    similarity_scores = []
    
    print(f"Memulai evaluasi terhadap {len(indices_sample)} sample lowongan...")
    
    for idx in indices_sample:
        row = df.iloc[idx]
        query_text = row["combined_text"]
        judul_query = str(row["job_title"]).lower().split()
        # Cari kata kunci utama (lewati kata depan jika ada)
        keywords = [w for w in judul_query if len(w) > 2]
        if not keywords:
            keywords = judul_query[:1]
            
        # Encode query
        query_vec = model.encode([query_text], normalize_embeddings=True).astype(np.float32)
        
        # Cari k+1 hasil terdekat
        sim_vals, search_indices = index.search(query_vec, k + 1)
        
        relevan = 0
        first_rank_relevan = 0
        count = 0
        
        for rank, res_idx in enumerate(search_indices[0]):
            # Lewati pencocokan dengan dirinya sendiri
            if res_idx == idx or res_idx >= len(df):
                continue
                
            count += 1
            judul_hasil = str(df.iloc[res_idx]["job_title"]).lower()
            
            # Cek apakah ada kata kunci query yang muncul di judul hasil
            is_relevan = any(kw in judul_hasil for kw in keywords)
            if is_relevan:
                relevan += 1
                if first_rank_relevan == 0:
                    first_rank_relevan = count
                    
            if count >= k:
                break
                
        # Precision@K = relevan / K
        precision_scores.append(relevan / k)
        
        # MRR = 1 / rank dari relevan pertama
        if first_rank_relevan > 0:
            mrr_scores.append(1.0 / first_rank_relevan)
        else:
            mrr_scores.append(0.0)
            
        # Simpan nilai similarity rata-rata teratas (index flat ip mengembalikan cosine similarity)
        similarity_scores.append(np.mean(sim_vals[0][1:k+1]))
        
    mean_precision = np.mean(precision_scores)
    mean_mrr = np.mean(mrr_scores)
    mean_sim = np.mean(similarity_scores)
    
    print("\n--- HASIL EVALUASI AKURASI MODEL ---")
    print(f"Rata-rata Precision@{k}       : {mean_precision:.2%}")
    print(f"Mean Reciprocal Rank (MRR) : {mean_mrr:.4f}")
    print(f"Rata-rata Cosine Similarity: {mean_sim:.4f}")
    print(f"Artinya: Dari {k} lowongan rekomendasi teratas, rata-rata {mean_precision*k:.1f} lowongan sangat relevan secara semantik.")

evaluasi_independen(df_eval, index_eval, model_eval, jumlah_sample=100, k=5)


Memuat model Sentence-BERT...
Memuat indeks FAISS dan metadata...
Memulai evaluasi terhadap 100 sample lowongan...

--- HASIL EVALUASI AKURASI MODEL ---
Rata-rata Precision@5       : 79.00%
Mean Reciprocal Rank (MRR) : 0.9153
Rata-rata Cosine Similarity: 0.7292
Artinya: Dari 5 lowongan rekomendasi teratas, rata-rata 4.0 lowongan sangat relevan secara semantik.
